# 01 · Explore the data

A first look at the four collections: how many images and patients each has, what the index
tables contain, what the images look like, and a first overlap question.

**How to run:** top right, click **Select Kernel** → choose **chest_overlap** (or the `.venv`
Python). Then press **Shift + Enter** to run one cell at a time, from top to bottom.

This notebook only *reads* data. It works with whatever is already downloaded and indexed
(see `docs/GETTING_STARTED.md`). Sections whose data is missing just print a note.

## 1. Setup
Import our package and see which data folder is used.

In [ ]:
import pandas as pd
from chest_overlap import COLLECTIONS, data_root, load_index, show_rows

ROOT = data_root()   # $CHEST_OVERLAP_DATA, else $HPCWORK/chest_overlap, else ./data
print("data folder:", ROOT)
print("collections:", list(COLLECTIONS))

## 2. Load the index tables
One row per image. `load_index()` stacks all tables that exist into one DataFrame
(a table, like an Excel sheet).

In [ ]:
available = [k for k in COLLECTIONS if (ROOT / "index" / f"{k}.csv").exists()]
print("indexed so far:", available or "nothing yet - run the download/index step first")

df = load_index(available, ROOT) if available else pd.DataFrame()
df.head()

## 3. Overview per collection
How many images are listed, how many are on disk, how many distinct patients, and how the
official split is divided.

In [ ]:
def overview(df):
    rows = []
    for key, g in df.groupby("collection"):
        rows.append({
            "collection": key,
            "images listed": len(g),
            "files on disk": (g["rel_path"] != "").sum(),
            "patients": g.loc[g["patient_id"] != "", "patient_id"].nunique(),
            "eval images": g["is_eval"].astype(str).eq("True").sum(),
            "splits": ", ".join(sorted(s for s in g["split"].unique() if s)) or "none",
        })
    return pd.DataFrame(rows).set_index("collection")

overview(df) if len(df) else "no data yet"

## 4. What columns does a table have?
The first seven columns are the same for every collection; the rest are extras
(see `docs/MANUAL.md`, section 6).

In [ ]:
for key, g in df.groupby("collection"):
    extras = [c for c in g.columns[7:] if g[c].notna().any() and g[c].astype(str).ne("").any()]
    print(f"{key:8s} extra columns: {extras}")

## 5. Look at some images
Shows the cached copy if it exists (fast), otherwise the original file.
Change `n` or the collection to see others.

In [ ]:
n = 4
for key, g in df.groupby("collection"):
    on_disk = g[g["rel_path"] != ""]
    if len(on_disk):
        print(key)
        show_rows(on_disk.sample(min(n, len(on_disk)), random_state=0))

## 6. Positive control: RSNA images and their ChestX-ray14 originals
RSNA published which NIH image each RSNA image came from. Each pair below is the *same*
radiograph released twice. Do they look identical? What changed?

In [ ]:
if {"rsna", "nih"} <= set(df["collection"]):
    rsna = df[(df["collection"] == "rsna") & (df["linked_nih_image_id"] != "") & (df["rel_path"] != "")]
    nih = df[df["collection"] == "nih"].set_index("image_id")
    pairs = rsna[rsna["linked_nih_image_id"].isin(nih.index[nih["rel_path"] != ""])].head(3)
    print(f"{(df['linked_nih_image_id'].fillna('') != '').sum()} RSNA images are linked to NIH")
    for _, r in pairs.iterrows():
        both = pd.DataFrame([r, nih.loc[r["linked_nih_image_id"]].rename(r["linked_nih_image_id"])])
        both["image_id"] = [r["image_id"], r["linked_nih_image_id"]]
        show_rows(both, cols=2)
else:
    print("needs both rsna and nih indexed")

## 7. A first overlap question: patients on both sides of the split
If one patient has images in the training half **and** the evaluation half, results measured on
that split are optimistic. This counts such patients using only the IDs the releases provide
(no image matching yet).

In [ ]:
def patients_across_split(g):
    g = g[g["patient_id"] != ""]
    if g.empty or g["split"].eq("").all():
        return None
    sides = g.groupby("patient_id")["is_eval"].agg(lambda s: set(s.astype(str)))
    both = sides[sides.map(len) == 2]
    eval_imgs = g[g["is_eval"].astype(str) == "True"]
    return {
        "patients with ids": len(sides),
        "patients on both sides": len(both),
        "eval images from those patients": eval_imgs["patient_id"].isin(both.index).sum(),
        "eval images total": len(eval_imgs),
    }

pd.DataFrame({k: patients_across_split(g) for k, g in df.groupby("collection")
              if patients_across_split(g)}).T

**Expected:** NIH should show 0 (its official split is patient-wise). Kermany may not:
that would be patient-level contamination found from file names alone.

## Next steps
* Copy this notebook to start your own: right-click → *Copy*, name it `02_<topic>.ipynb`.
* Before committing a notebook: **Clear All Outputs** (toolbar ⋯), so the file stays small.